# Sending tool results

Claude asked for a tool in the previous lesson. Now your code runs the function and sends the
result back, and Claude writes the final answer. The whole round trip is 4 messages in history:

| # | role | content |
|---|---|---|
| 1 | user | "What is the exact time?" |
| 2 | assistant | `tool_use` block: call `get_current_datetime` with these arguments, id `toolu_...` |
| 3 | user | `tool_result` block: the function output for id `toolu_...` |
| 4 | assistant | text: "It is 14:41:05" |

The `tool_result` block goes into a **user** message:

| Field | Value |
|---|---|
| `type` | `"tool_result"` |
| `tool_use_id` | the `id` of the `tool_use` block it answers |
| `content` | the function output as a string |
| `is_error` | `True` if the function failed, `False` otherwise |

`tool_use_id` matters when Claude asks for several tools at once: the ids tell it which result
answers which request. The follow-up request still sends `tools=[...]`: the history mentions the
tool, and Claude needs its schema to make sense of it.

## Setup

The function, schema and helpers from the previous lessons.

In [1]:
from datetime import datetime

from anthropic import Anthropic
from anthropic.types import Message, ToolParam
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time formatted according to the specified format. "
        "Use it whenever the answer depends on the current date or time, for example to "
        "compute a date relative to today. "
        "Returns the formatted date and time as a string."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A string specifying the format of the returned datetime. "
                    "Uses Python's strftime format codes, e.g. '%Y-%m-%d' or '%H:%M'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

## Step 1: ask, Claude requests the tool

In [2]:
messages = []
add_user_message(messages, "What is the exact time, formatted as HH:MM:SS?")

response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message(messages, response)

print("stop_reason:", response.stop_reason)
print("blocks:", [block.type for block in response.content])

stop_reason: tool_use
blocks: ['tool_use']


## Step 2: find the `tool_use` block

The lesson reads `response.content[1]` because its reply starts with a text block. Without that text
the tool request is `content[0]`. Searching by type works in both cases.

In [3]:
tool_use = next(block for block in response.content if block.type == "tool_use")

print("id:   ", tool_use.id)
print("name: ", tool_use.name)
print("input:", tool_use.input)

id:    toolu_bdrk_01N3s215DTscVUqNmDbrx41o
name:  get_current_datetime
input: {'date_format': '%H:%M:%S'}


## Step 3: run the function

`input` is a dict of arguments, `**` unpacks it into keyword arguments.

In [4]:
result = get_current_datetime(**tool_use.input)
print(result)

17:46:41


## Step 4: send the result back as a user message

In [5]:
add_user_message(messages, [
    {
        "type": "tool_result",
        "tool_use_id": tool_use.id,
        "content": result,
        "is_error": False,
    }
])

## Step 5: Claude writes the final answer

In [6]:
response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message(messages, response)

print("stop_reason:", response.stop_reason)
print(text_from_message(response))

stop_reason: end_turn
The exact time is **17:46:41** (5:46:41 PM).


## The whole history

In [7]:
for number, message in enumerate(messages, start=1):
    content = message["content"]
    if isinstance(content, str):
        summary = repr(content)
    else:
        summary = ", ".join(
            block["type"] if isinstance(block, dict) else block.type for block in content
        )
    print(f"{number}. {message['role']:9} {summary}")

1. user      'What is the exact time, formatted as HH:MM:SS?'
2. assistant tool_use
3. user      tool_result
4. assistant text


Here every step is written by hand for one tool call. The next lessons put it in a loop:
call Claude, run every requested tool, send the results, repeat until `stop_reason` is not `tool_use`.